In [5]:

import pandas as pd

df = pd.read_parquet("eda_hourly_history.parquet")

print(df.shape)


(19864, 18)


### Step 1: Feature Selection

**Objective:** Select relevant input features for hourly electricity consumption forecasting.

**Selected Features:**
- `hour`: Captures hourly consumption patterns.
- `dayofweek`: Captures weekly consumption patterns.
- `month`: Captures seasonal variations.
- `is_weekend`: Distinguishes weekends from weekdays.

**Reason:** These features are known in advance and can be used for future predictions without data leakage.

**Output:** A DataFrame containing the four selected input features.


In [6]:
selected_features=[
    'hour',
    "dayofweek",
    'month',
    'is_weekend'

]
x = df[selected_features]
print(x.head())
print("shape:",x.shape)

   hour  dayofweek  month  is_weekend
0     9          2     11       False
1    10          2     11       False
2    11          2     11       False
3    12          2     11       False
4    13          2     11       False
shape: (19864, 4)



### Step 1.1: Feature Identification and Leakage Assessment

**Objective:** Identify the data types and roles of the original dataset columns.

**Method:** Classify columns as selected features, target, metadata, or excluded measurements.

**Reason:** Only information available before prediction should be used as model input.


In [20]:

# Identify feature types and leakage risks

future_columns = [
    "energy_kwh_observed",
    "energy_kwh_complete_homes",
    "homes_reporting",
    "homes_complete",
    "readings_observed",
    "readings_expected_in_span",
    "coverage_pct_in_span"
]

feature_info = []

for column in df.columns:

    if column in selected_features:
        role = "Selected Feature"
        risk = "No"

    elif column == "avg_kwh_per_complete_home":
        role = "Proposed Target"
        risk = "Yes, if used in X"

    elif column in future_columns:
        role = "Observed / Quality Data"
        risk = "Risk if used for future hours"

    elif column == "hour_start":
        role = "Timestamp"
        risk = "No"

    else:
        role = "Excluded Calendar Feature"
        risk = "No"

    feature_info.append({
        "Feature": column,
        "Data Type": str(df[column].dtype),
        "Role": role,
        "Leakage Risk": risk
    })

feature_info_df = pd.DataFrame(feature_info)

display(feature_info_df.set_index("Feature"))


,Data Type,Role,Leakage Risk
Feature,,,
hour_start,datetime64[us],Timestamp,No
energy_kwh_observed,float64,Observed / Quality Data,Risk if used for future hours
energy_kwh_complete_homes,float64,Observed / Quality Data,Risk if used for future hours
homes_reporting,int64,Observed / Quality Data,Risk if used for future hours
homes_complete,int64,Observed / Quality Data,Risk if used for future hours
readings_observed,int64,Observed / Quality Data,Risk if used for future hours
readings_expected_in_span,int64,Observed / Quality Data,Risk if used for future hours
coverage_pct_in_span,float64,Observed / Quality Data,Risk if used for future hours
avg_kwh_per_complete_home,float64,Proposed Target,"Yes, if used in X"



### Step 2: Feature Creation

**Objective:** Create new time-based features to capture daily, weekly, and seasonal patterns in electricity consumption.

**Method:** Cyclical Encoding using sine and cosine transformations.

**New Features:**
- `hour_sin`, `hour_cos`: Daily patterns.
- `dow_sin`, `dow_cos`: Weekly patterns.
- `month_sin`, `month_cos`: Seasonal patterns.

**Reason:** Time-based variables are cyclical. These transformations preserve their periodic relationships and use only calendar information, avoiding target leakage.

**Output:** Six new features created from existing calendar columns.


In [7]:

import numpy as np

# Daily cycle (24 hours)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Weekly cycle (7 days)
df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

# Yearly cycle (12 months)
df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

# Select final candidate features
feature_columns = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "month_sin", "month_cos",
    "is_weekend"
]

X = df[feature_columns].copy()

print(X.head())
print("\nX Shape:", X.shape)


       hour_sin  hour_cos   dow_sin   dow_cos  month_sin  month_cos  \
0  7.071068e-01 -0.707107  0.974928 -0.222521  -0.866025        0.5   
1  5.000000e-01 -0.866025  0.974928 -0.222521  -0.866025        0.5   
2  2.588190e-01 -0.965926  0.974928 -0.222521  -0.866025        0.5   
3  1.224647e-16 -1.000000  0.974928 -0.222521  -0.866025        0.5   
4 -2.588190e-01 -0.965926  0.974928 -0.222521  -0.866025        0.5   

   is_weekend  
0       False  
1       False  
2       False  
3       False  
4       False  

X Shape: (19864, 7)



### Step 3: Review Feature Changes

**Objective:** Document the selected, created, and excluded features.

**Created Features:**
- `hour_sin`, `hour_cos`
- `dow_sin`, `dow_cos`
- `month_sin`, `month_cos`

**Retained Feature:**
- `is_weekend`

**Transformed Features:**
- `hour`, `dayofweek`, and `month` were used to create cyclical features.

**Excluded from Model Inputs:**
- Redundant calendar features.
- Same-hour energy measurements and reporting statistics.
- The proposed target variable.

**Reason:** Keep the initial feature set simple, relevant, and safe from future data leakage.

**Result:** Seven candidate input features for the forecasting task.


In [8]:
from IPython.display import display

feature_review = pd.DataFrame({
    "Features": [
        "hour_sin, hour_cos",
        "dow_sin, dow_cos",
        "month_sin, month_cos",
        "is_weekend",
        "hour, dayofweek, month",
        "Other calendar columns",
        "Energy & coverage columns",
        "avg_kwh_per_complete_home"
    ],
    "Action": [
        "Created",
        "Created",
        "Created",
        "Kept",
        "Transformed",
        "Excluded",
        "Excluded",
        "Excluded"
    ],
    "Reason": [
        "Represent daily patterns",
        "Represent weekly patterns",
        "Represent seasonal patterns",
        "Identify weekends",
        "Replaced by cyclical features",
        "Not needed in the initial X",
        "Future values are unavailable",
        "Proposed prediction target"
    ]
})

display(feature_review)


,Features,Action,Reason
0,"hour_sin, hour_cos",Created,Represent daily patterns
1,"dow_sin, dow_cos",Created,Represent weekly patterns
2,"month_sin, month_cos",Created,Represent seasonal patterns
3,is_weekend,Kept,Identify weekends
4,"hour, dayofweek, month",Transformed,Replaced by cyclical features
5,Other calendar columns,Excluded,Not needed in the initial X
6,Energy & coverage columns,Excluded,Future values are unavailable
7,avg_kwh_per_complete_home,Excluded,Proposed prediction target



## Part 2: Data Splitting

### Step 1: Define Inputs (X) and Target (y)

**Objective:** Define the input features and the prediction target for hourly electricity consumption forecasting.

**Input Features (X):**
- hour_sin, hour_cos
- dow_sin, dow_cos
- month_sin, month_cos
- is_weekend

**Proposed Target (y):**
- avg_kwh_per_complete_home

**Reason:** The selected features represent calendar patterns and are available before making future predictions.

The proposed target represents average hourly energy consumption per complete-reporting household.

**Expected Output:** A feature matrix (X) and a target vector (y), ready for temporal splitting after target validation.


In [9]:
target_column = "avg_kwh_per_complete_home"

# Select input features
X = df[feature_columns].copy()

# Select target values
y = df[target_column].copy()

# Display results
print("X Shape:", X.shape)
print("y Shape:", y.shape)
print("Missing Target Values:", y.isna().sum())

print("\nInput Features (X):")
display(X.head())

print("\nTarget (y):")
display(y.head().to_frame(name=target_column))


X Shape: (19864, 7)
y Shape: (19864,)
Missing Target Values: 1

Input Features (X):


,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,is_weekend
0,7.071068e-01,-0.707107,0.974928,-0.222521,-0.866025,0.5,False
1,5.000000e-01,-0.866025,0.974928,-0.222521,-0.866025,0.5,False
2,2.588190e-01,-0.965926,0.974928,-0.222521,-0.866025,0.5,False
3,1.224647e-16,-1.000000,0.974928,-0.222521,-0.866025,0.5,False
4,-2.588190e-01,-0.965926,0.974928,-0.222521,-0.866025,0.5,False



Target (y):


,avg_kwh_per_complete_home
0,0.565000
1,0.251333
2,0.303286
3,0.719333
4,0.483750



### Step 2: Temporal Data Splitting

**Objective:** Split the hourly dataset chronologically into training, validation, and testing sets.

**Splitting Strategy:** Time-based splitting without random shuffling.

**Proposed Periods:**
- Training: July 2012 – June 2013
- Validation: July 2013 – August 2013
- Testing: September 2013 – February 27, 2014

**Reason:** Electricity consumption is time-dependent. Chronological splitting ensures that training observations occur before validation and testing observations.

**Output:** Separate X and y datasets for training, validation, and testing.

**Note:** The forecasting target and split dates are provisional and subject to academic review.


In [10]:
data = df.copy()
data["hour_start"] = pd.to_datetime(data["hour_start"])
data = data.sort_values("hour_start").reset_index(drop=True)

# 2. Split the dataset by date
train_df = data[
    (data["hour_start"] >= "2012-07-01") &
    (data["hour_start"] < "2013-07-01")
].copy()

val_df = data[
    (data["hour_start"] >= "2013-07-01") &
    (data["hour_start"] < "2013-09-01")
].copy()

test_df = data[
    (data["hour_start"] >= "2013-09-01") &
    (data["hour_start"] < "2014-02-28")
].copy()

# 3. Separate features (X) and target (y)
X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_val = val_df[feature_columns].copy()
y_val = val_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

# 4. Display a summary table
split_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Testing"],
    "Rows": [len(train_df), len(val_df), len(test_df)],
    "Features": [
        X_train.shape[1],
        X_val.shape[1],
        X_test.shape[1]
    ],
    "Missing Target": [
        y_train.isna().sum(),
        y_val.isna().sum(),
        y_test.isna().sum()
    ]
})

display(split_summary)


,Dataset,Rows,Features,Missing Target
0,Training,8760,7,0
1,Validation,1488,7,0
2,Testing,4320,7,0



### Step 3: Avoid Data Leakage

**Objective:** Validate the chronological data split and prevent information leakage.

**Validation Checks:**
1. Ensure training data comes before validation data.
2. Ensure validation data comes before testing data.
3. Verify that the target is not included in input features.
4. Check for missing values in X and y.
5. Check for duplicate timestamps across the splits.

**Reason:** These checks help ensure that future information is not incorrectly used during model preparation.

**Expected Output:** A validation table showing the result of each check.


In [11]:


train_before_val = (
    train_df["hour_start"].max() <
    val_df["hour_start"].min()
)

val_before_test = (
    val_df["hour_start"].max() <
    test_df["hour_start"].min()
)

# Check that the target is not an input feature
target_excluded = target_column not in feature_columns

# Check missing values in features
no_missing_X = (
    X_train.notna().all().all()
    and X_val.notna().all().all()
    and X_test.notna().all().all()
)

# Check missing target values
no_missing_y = (
    y_train.notna().all()
    and y_val.notna().all()
    and y_test.notna().all()
)

# Check duplicate timestamps
all_timestamps = pd.concat([
    train_df["hour_start"],
    val_df["hour_start"],
    test_df["hour_start"]
])

no_duplicates = all_timestamps.is_unique

# Display validation results
validation_results = pd.DataFrame({
    "Validation Check": [
        "Training before Validation",
        "Validation before Testing",
        "Target excluded from X",
        "No missing features",
        "No missing targets",
        "No duplicate timestamps"
    ],
    "Status": [
        train_before_val,
        val_before_test,
        target_excluded,
        no_missing_X,
        no_missing_y,
        no_duplicates
    ]
})

validation_results["Status"] = validation_results[
    "Status"
].map({True: "PASS", False: "FAIL"})

display(validation_results.set_index("Validation Check"))


,Status
Validation Check,
Training before Validation,PASS
Validation before Testing,PASS
Target excluded from X,PASS
No missing features,PASS
No missing targets,PASS
No duplicate timestamps,PASS



## Part 3: Data Preprocessing

### Step 1: Prepare Input Features

**Objective:** Prepare the selected features for future machine learning.

**Preprocessing Decisions:**
- Missing Values: No imputation if previous checks pass.
- Encoding: Convert `is_weekend` into a numeric binary feature.
- Scaling: Not applied to the initial calendar-based features.

**Reason:** The six cyclical features are already numerical and bounded between -1 and 1.

**Output:** Prepared Train, Validation, and Test feature sets.


In [12]:
assert no_missing_X and no_missing_y, (
    "Missing values detected. Review before preprocessing."
)

# Create separate prepared feature sets
X_train_prepared = X_train.copy()
X_val_prepared = X_val.copy()
X_test_prepared = X_test.copy()

# Convert weekend indicator to numeric format
for dataset in [X_train_prepared, X_val_prepared, X_test_prepared]:
    dataset["is_weekend"] = dataset["is_weekend"].astype(int)

# Document preprocessing decisions
preprocessing_summary = pd.DataFrame({
    "Step": [
        "Missing Values",
        "Categorical Encoding",
        "Feature Scaling"
    ],
    "Decision": [
        "No imputation needed",
        "Weekend converted to integer",
        "Not applied to current features"
    ]
})

display(preprocessing_summary.set_index("Step"))


,Decision
Step,
Missing Values,No imputation needed
Categorical Encoding,Weekend converted to integer
Feature Scaling,Not applied to current features



### Step 2: Feature Scaling

**Objective:** Standardize the six numerical cyclical features using StandardScaler.

**Method:**
- Fit StandardScaler on the training data only.
- Apply the same scaler to validation and testing data.
- Keep `is_weekend` as a binary feature (0 or 1).

**Reason:** Standardization gives numerical features comparable scales without using validation or testing statistics.

**Output:** Scaled training, validation, and testing feature sets.


In [13]:

from sklearn.preprocessing import StandardScaler

# Features to scale
scaling_columns = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "month_sin", "month_cos"
]

# Create the scaler
scaler = StandardScaler()

# Fit only on training data
X_train_prepared[scaling_columns] = scaler.fit_transform(
    X_train_prepared[scaling_columns]
)

# Apply the same scaler to validation and testing
X_val_prepared[scaling_columns] = scaler.transform(
    X_val_prepared[scaling_columns]
)

X_test_prepared[scaling_columns] = scaler.transform(
    X_test_prepared[scaling_columns]
)

# Display scaled features
display(X_train_prepared.head())


,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,is_weekend
5295,2.150812e-17,1.414214,-1.102317,0.879603,0.004913,-1.407534,1
5296,3.660254e-01,1.366025,-1.102317,0.879603,0.004913,-1.407534,1
5297,7.071068e-01,1.224745,-1.102317,0.879603,0.004913,-1.407534,1
5298,1.000000e+00,1.000000,-1.102317,0.879603,0.004913,-1.407534,1
5299,1.224745e+00,0.707107,-1.102317,0.879603,0.004913,-1.407534,1



## Part 4: Final Data Checks

### Step 1: Validate Prepared Data

**Objective:** Verify that the prepared datasets are consistent and contain valid values.

**Validation Checks:**
- Consistent feature columns across all splits.
- No missing or infinite values.
- Valid binary weekend values (0 or 1).
- Correct alignment between X and y.

**Expected Output:** A validation summary for Training, Validation, and Testing.

**Note:** These checks are performed after preprocessing.


In [14]:
datasets = {
    "Training": (X_train_prepared, y_train),
    "Validation": (X_val_prepared, y_val),
    "Testing": (X_test_prepared, y_test)
}

results = []

for name, (X_part, y_part) in datasets.items():

    # Check feature columns
    columns_ok = list(X_part.columns) == feature_columns

    # Check missing or infinite values
    features_ok = np.isfinite(
        X_part.to_numpy(dtype=float)
    ).all()

    target_ok = np.isfinite(
        y_part.to_numpy(dtype=float)
    ).all()

    # Check weekend encoding
    weekend_ok = X_part["is_weekend"].isin([0, 1]).all()

    # Check X and y alignment
    alignment_ok = X_part.index.equals(y_part.index)

    results.append({
        "Dataset": name,
        "Rows": len(X_part),
        "Features": X_part.shape[1],
        "Columns": "PASS" if columns_ok else "FAIL",
        "X Values": "PASS" if features_ok else "FAIL",
        "y Values": "PASS" if target_ok else "FAIL",
        "Weekend": "PASS" if weekend_ok else "FAIL",
        "Alignment": "PASS" if alignment_ok else "FAIL"
    })

validation_summary = pd.DataFrame(results)

display(validation_summary.set_index("Dataset"))


,Rows,Features,Columns,X Values,y Values,Weekend,Alignment
Dataset,,,,,,,
Training,8760,7,PASS,PASS,PASS,PASS,PASS
Validation,1488,7,PASS,PASS,PASS,PASS,PASS
Testing,4320,7,PASS,PASS,PASS,PASS,PASS



### Step 3: Validate 24-Hour Forecasting Readiness

**Objective:** Check whether each data split supports complete 24-hour forecasting periods.

**Method:**
- Group hourly observations by calendar day.
- Check that each day contains 24 hourly timestamps.
- Verify that all 24 target values are available.

**Reason:** Our forecasting objective is to predict 24 hourly consumption values for each forecast day.

**Expected Output:** A summary of valid forecasting days in Training, Validation, and Testing.


In [15]:
splits = {
    "Training": train_df,
    "Validation": val_df,
    "Testing": test_df
}

forecast_summary = []

for name, dataset in splits.items():

    daily = dataset.copy()

    # Identify the calendar day of each hour
    daily["forecast_day"] = daily["hour_start"].dt.normalize()

    # Count hourly timestamps and available targets per day
    daily_counts = daily.groupby("forecast_day").agg(
        hours=("hour_start", "nunique"),
        valid_targets=(target_column, "count")
    )

    # Verify that timestamps are aligned to whole hours
    hourly_grid_ok = (
        daily["hour_start"].dt.minute.eq(0)
        & daily["hour_start"].dt.second.eq(0)
    ).all()

    # A valid day must have 24 hours and 24 target values
    valid_days = (
        (daily_counts["hours"] == 24)
        & (daily_counts["valid_targets"] == 24)
    )

    ready_days = int(valid_days.sum()) if hourly_grid_ok else 0

    forecast_summary.append({
        "Dataset": name,
        "Total Days": len(daily_counts),
        "Valid 24h Days": ready_days,
        "Status": (
            "PASS" if ready_days == len(daily_counts)
            else "REVIEW"
        )
    })

forecast_results = pd.DataFrame(forecast_summary)

display(forecast_results.set_index("Dataset"))


,Total Days,Valid 24h Days,Status
Dataset,,,
Training,365,365,PASS
Validation,62,62,PASS
Testing,180,180,PASS



### Step 4: Save Prepared Data

**Objective:** Save the feature-engineered dataset and the final prepared data splits in Parquet format.

**Outputs:**
- Feature-engineered hourly dataset.
- Prepared training dataset.
- Prepared validation dataset.
- Prepared testing dataset.
- Fitted StandardScaler for future use.

**Reason:** Preserve the processed data and preprocessing configuration for reproducible machine learning experiments.

**Note:** The original source dataset remains unchanged.


In [19]:
output_file = "eda_hourly_history_features.parquet"

df.to_parquet(output_file, index=False)

# Verify the saved file
saved_df = pd.read_parquet(output_file)

print("Saved Successfully!")
print("Dataset Shape:", saved_df.shape)

display(saved_df[[
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "month_sin", "month_cos"
]].head())


Saved Successfully!
Dataset Shape: (19864, 24)


,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
0,7.071068e-01,-0.707107,0.974928,-0.222521,-0.866025,0.5
1,5.000000e-01,-0.866025,0.974928,-0.222521,-0.866025,0.5
2,2.588190e-01,-0.965926,0.974928,-0.222521,-0.866025,0.5
3,1.224647e-16,-1.000000,0.974928,-0.222521,-0.866025,0.5
4,-2.588190e-01,-0.965926,0.974928,-0.222521,-0.866025,0.5



### Step 5: Save Final Prepared Datasets

**Objective:** Save the processed train, validation, and test datasets and the fitted preprocessing configuration.

**Method:** Export prepared data to Parquet and save the training-fitted StandardScaler.

**Expected Output:** Three prepared Parquet files and one preprocessing bundle for future model training.


In [21]:

from pathlib import Path
import joblib

# Create output directory
output_dir = Path("data/ml/task2")
output_dir.mkdir(parents=True, exist_ok=True)

# Prepared datasets
prepared_splits = {
    "train": (train_df, X_train_prepared, y_train),
    "validation": (val_df, X_val_prepared, y_val),
    "test": (test_df, X_test_prepared, y_test)
}

saved_results = []

for name, (source, features, target) in prepared_splits.items():

    # Verify row alignment
    assert source.index.equals(features.index)
    assert features.index.equals(target.index)

    # Combine timestamp, features, and target
    prepared_df = pd.concat([
        source[["hour_start"]],
        features,
        target.rename(target_column)
    ], axis=1)

    file_path = output_dir / f"{name}_prepared.parquet"
    prepared_df.to_parquet(file_path, index=False)

    # Read back to verify successful saving
    verified = pd.read_parquet(file_path)

    assert verified.shape == prepared_df.shape
    assert verified.columns.tolist() == prepared_df.columns.tolist()

    saved_results.append({
        "Dataset": name,
        "Rows": len(verified),
        "Columns": verified.shape[1],
        "Status": "SAVED & VERIFIED"
    })

# Save the fitted scaler and feature configuration
joblib.dump({
    "scaler": scaler,
    "scaled_columns": scaling_columns,
    "feature_columns": feature_columns
}, output_dir / "preprocessing_bundle.joblib")

display(pd.DataFrame(saved_results).set_index("Dataset"))

print("Output Directory:", output_dir.resolve())


,Rows,Columns,Status
Dataset,,,
train,8760,9,SAVED & VERIFIED
validation,1488,9,SAVED & VERIFIED
test,4320,9,SAVED & VERIFIED


Output Directory: D:\big data and machine learning project\data\ml\task2
